### import packages

In [1]:
import openai
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

### create qdrant collection for hybrid search
previous qdrant db use cosine similarity to create a collection, since we are going to run hydrib search, so we need to create a new qdrant collection

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [3]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-hybrid-search",
    # name the vector field as "text-embedding-3-small", before we did not name it as we only have one vector field
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF)
    }
)

True

In [4]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search",
    field_name = 'parent_asin',
    field_schema = PayloadSchemaType.KEYWORD # run keyword search against parent_asin
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### embedding function

In [36]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [37]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with Amazon inventory data

In [7]:
df_items = pd.read_json("../../data/raw/meta_Electronics_2022_2023_ratings_100_sample_1000.jsonl", lines=True)

In [8]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Computers,Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lig...,4.5,153,[Specifically Designed For: Samsung Galaxy Tab...,[Specifically design for Samsung Galaxy Tab E ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Leafbook,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '7 Inches', '...",B01DJI7TP8,NaN,NaN,NaN
1,All Electronics,Lume Cube AIR Video Conference Lighting Kit - ...,3.7,215,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LUME CUBE,"[Electronics, Camera & Photo, Lighting & Studi...",{'Product Dimensions': '1.61 x 1.61 x 1.1 inch...,B07M6SWKLT,NaN,NaN,NaN
2,All Electronics,Ultimate Ears BOOM 2 Portable Waterproof & Sho...,4.5,1278,[Insanely good 360˚ sound - Ultimate Ears Boom...,[Ultimate Ears boom 2 is the 360-degree wirele...,68.49,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'HONEST review of the UE Boom 2 spe...,Ultimate Ears,"[Electronics, Portable Audio & Video, Portable...",{'Product Dimensions': '2.75 x 2.75 x 7.08 inc...,B07J31M7PW,NaN,NaN,NaN
3,All Electronics,Seagate FreeAgent GoFlex TV STAJ100,3.5,108,[Watch streaming media directly from the Inter...,"[Product Description, The Seagate GoFlex TV HD...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Seagate,"[Electronics, Television & Video, Streaming Me...","{'Brand Name': 'Seagate', 'Item Weight': '10.7...",B003IT6YFK,NaN,NaN,NaN
4,All Electronics,"JSAUX RCA Cable（10ft/3M, RCA Audio Cable Shiel...",4.5,110,[🎧【High-quality Stereo Sound】Built-in multiple...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'J&D 2RCA Male to 2RCA Male Cable',...",JSAUX,"[Electronics, Television & Video, Accessories,...",{'Package Dimensions': '10.16 x 3.15 x 0.71 in...,B099ZPY68J,NaN,NaN,NaN


### Preprocess title and features

In [9]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

def extract_first_large_image(row):
    return row["images"][0].get("large", "")

df_items["preprocessed_description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,preprocessed_description,image
0,Computers,Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lig...,4.5,153,[Specifically Designed For: Samsung Galaxy Tab...,[Specifically design for Samsung Galaxy Tab E ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Leafbook,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '7 Inches', '...",B01DJI7TP8,NaN,NaN,NaN,Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lig...,https://m.media-amazon.com/images/I/51PML408lo...
1,All Electronics,Lume Cube AIR Video Conference Lighting Kit - ...,3.7,215,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LUME CUBE,"[Electronics, Camera & Photo, Lighting & Studi...",{'Product Dimensions': '1.61 x 1.61 x 1.1 inch...,B07M6SWKLT,NaN,NaN,NaN,Lume Cube AIR Video Conference Lighting Kit - ...,https://m.media-amazon.com/images/I/411wKstRw1...
2,All Electronics,Ultimate Ears BOOM 2 Portable Waterproof & Sho...,4.5,1278,[Insanely good 360˚ sound - Ultimate Ears Boom...,[Ultimate Ears boom 2 is the 360-degree wirele...,68.49,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'HONEST review of the UE Boom 2 spe...,Ultimate Ears,"[Electronics, Portable Audio & Video, Portable...",{'Product Dimensions': '2.75 x 2.75 x 7.08 inc...,B07J31M7PW,NaN,NaN,NaN,Ultimate Ears BOOM 2 Portable Waterproof & Sho...,https://m.media-amazon.com/images/I/41kYEBxNW2...
3,All Electronics,Seagate FreeAgent GoFlex TV STAJ100,3.5,108,[Watch streaming media directly from the Inter...,"[Product Description, The Seagate GoFlex TV HD...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Seagate,"[Electronics, Television & Video, Streaming Me...","{'Brand Name': 'Seagate', 'Item Weight': '10.7...",B003IT6YFK,NaN,NaN,NaN,Seagate FreeAgent GoFlex TV STAJ100 Watch stre...,https://m.media-amazon.com/images/I/31Vy2ogKvD...
4,All Electronics,"JSAUX RCA Cable（10ft/3M, RCA Audio Cable Shiel...",4.5,110,[🎧【High-quality Stereo Sound】Built-in multiple...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'J&D 2RCA Male to 2RCA Male Cable',...",JSAUX,"[Electronics, Television & Video, Accessories,...",{'Package Dimensions': '10.16 x 3.15 x 0.71 in...,B099ZPY68J,NaN,NaN,NaN,"JSAUX RCA Cable（10ft/3M, RCA Audio Cable Shiel...",https://m.media-amazon.com/images/I/41EbtDHhNE...


In [12]:
df_items.shape

(1000, 18)

In [10]:
list(df_items["preprocessed_description"].items())[0]

(0,
 "Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lightweight Shell Case for Samsung Galaxy Tab E lite 7.0 & Tab 3 Lite T110/T111, Black Specifically Designed For: Samsung Galaxy Tab 3 Lite 7.0 7'' T110 T111 Tablet / Galaxy Tab E Lite 7.0 (NOT FIT FOR Samsung Galaxy Tab 3 7.0 T210) Premium PU leather exterior and microfiber interior. Ultra slim and lightweight hard back design adds minimal bulk while protecting your precious device. Precise cut-outs and openings for easy access to all tablet features. Cover has flip capability to transform the case into a viewing stand and keyboard stand. One piece case, the front and back does not separate. Various colors are available.Magnetic closure.")

In [11]:
list(df_items["image"].items())[0]

(0, 'https://m.media-amazon.com/images/I/51PML408loL._AC_.jpg')

In [14]:
df_data_to_embed = df_items[["preprocessed_description", "image", "rating_number", "price", "average_rating", "parent_asin"]]
data_to_embed = df_data_to_embed.to_dict(orient="records")

In [16]:
len(data_to_embed)

1000

In [15]:
text_to_embed = [item["preprocessed_description"] for item in data_to_embed]

In [19]:
embeddings = get_embeddings_batch(text_to_embed)

Processed 100 of 1000
Processed 200 of 1000
Processed 300 of 1000
Processed 400 of 1000
Processed 500 of 1000
Processed 600 of 1000
Processed 700 of 1000
Processed 800 of 1000
Processed 900 of 1000
Processed 1000 of 1000


In [20]:
len(embeddings)

1000

In [28]:
pointstructs = []
i= 1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(PointStruct(
        id=i,
        vector={"text-embedding-3-small": embedding, 
        "bm25": 
            Document(text=data["preprocessed_description"], model="qdrant/bm25")}, # qdrant handle it natively
        payload=data
    ))
    i+=1


In [29]:
pointstructs[0].vector

{'text-embedding-3-small': [0.027679443359375,
  -0.06182861328125,
  -0.01552581787109375,
  0.0121307373046875,
  -0.0174407958984375,
  -0.0278778076171875,
  0.0120391845703125,
  0.040130615234375,
  0.0382080078125,
  -0.022430419921875,
  -0.0302886962890625,
  -0.01849365234375,
  -0.0174713134765625,
  0.0258026123046875,
  0.00899505615234375,
  0.03125,
  0.0712890625,
  -0.020538330078125,
  -0.024505615234375,
  0.0631103515625,
  -0.003330230712890625,
  0.01030731201171875,
  0.03973388671875,
  0.00672149658203125,
  -0.01453399658203125,
  -0.04595947265625,
  0.00635528564453125,
  -0.0369873046875,
  0.0023345947265625,
  0.01108551025390625,
  0.04071044921875,
  -0.0169830322265625,
  0.032440185546875,
  -0.0782470703125,
  -0.049835205078125,
  -0.0017843246459960938,
  0.036224365234375,
  -0.038726806640625,
  -0.01500701904296875,
  0.045989990234375,
  -0.0005006790161132812,
  0.0462646484375,
  0.035614013671875,
  0.0180816650390625,
  0.007656097412109375

In [30]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[:500], # sometimes time out if we add all points at once
    wait=True
)


UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [32]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[500:], # sometimes time out if we add all points at once
    wait=True
)


UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

### hybrid-retrieval

In [40]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20 # why 20? limit is often 3x - 10x of k
            ),
            Prefetch(
                query=Document(
                    text=query, #bm search need actual query text
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"), # reciprocal rank fusion
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [41]:
retrieve_data("can i get a tablet", k=20)

{'retrieved_context_ids': ['B08Z3N1YTR',
  'B07Q38R2XF',
  'B0BGHMDZHG',
  'B08DNC3L4M',
  'B07ZNPPXN6',
  'B00Y1NGRM2',
  'B0788JXFXB',
  'B07L8HX46F',
  'B07KFXQFQ6',
  'B081122SFR',
  'B073R8PCB2',
  'B08DMY7S84',
  'B004M5H6D8',
  'B09KY4D7RJ',
  'B00B3507Q8',
  'B0829D5423',
  'B0BNJ1QQCM',
  'B07RW5XZSH',
  'B09G2NKNW4',
  'B08HRVCLXL'],
 'retrieved_context': ["Short Micro USB Cable,Android Charger Compatible with Fire Stick,Roku Stick,PS4,Xbox,Fast Charging Cord Compatible with Kindle Tablet,Bose SoundLink Speaker,Chromecast,Computer Phone,Power Bank,8 inch 【Short Micro USB Cable for TV Stick or Other Devcies】 8 inches Micro USB Cable is perfect lenght for charging Android phones and Windows smart phones, tablets & MP3 players, digital cameras, camcorders & more. This short Android Charger Cord is Perfect length for connect TV stick from back TV USB port. 【Fast Charging and Data Sync】22 AWG wire, charge up to 30% faster than most standard alternatives.This 8 inch short Micro USB

In [ ]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20 # why 20? limit is often 3x - 10x of k
            ),
            Prefetch(
                query=Document(
                    text=query, #bm search need actual query text
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3, 1])), # should be tuned, based on the data; contextual vs keyword search
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [50]:
retrieve_data("can i get a tablet device", k=20)

{'retrieved_context_ids': ['B07Q38R2XF',
  'B0BGHMDZHG',
  'B081122SFR',
  'B08DMY7S84',
  'B00Y1NGRM2',
  'B0829D5423',
  'B07L8HX46F',
  'B004M5H6D8',
  'B07YQW55CN',
  'B07RW5XZSH',
  'B08Z44LXYL',
  'B0128Z5BPI',
  'B08Y55D9JM',
  'B07FSHNRXX',
  'B00B3507Q8',
  'B00QWQQHOE',
  'B08C2DVS1V',
  'B012Q066SG',
  'B07H9NGGYR',
  'B073Y1LPJR'],
 'retrieved_context': ['ASURION 2 Year Tablet Accident Protection Plan ($30 - $39.99) NO ADDITIONAL COST: You pay $0 for repairs – parts, labor and shipping included. COVERAGE: Plan starts on the date of purchase. Drops, spills and cracked screens due to normal use are covered from day one. Malfunctions are covered after the manufacturer’s warranty. EASY CLAIMS PROCESS: File a claim anytime online at www.Asurion.com/Amazon or by phone. Most claims approved within minutes. We will send you an Amazon e-gift card for the purchase price of your covered product. In some cases, we will replace or repair it. EXPERT TECH HELP: Real experts are available 